In [0]:
%pip install mlflow
dbutils.library.restartPython()

## Local testing of async calls
Testing two things mlflow trace and nested async functions

In [0]:
import mlflow
experiment = mlflow.set_experiment('/Users/jon.cheung@databricks.com/Alaska/mlflow-chain-tracing')
experiment.experiment_id

In [0]:
import asyncio
@mlflow.trace(span_type="func")
async def call_one(input_str: str):
  await asyncio.sleep(2)
  return f"call_one: {input_str}"

@mlflow.trace(span_type="func")
async def call_two(input_str: str):
  await asyncio.sleep(1)
  return f"call_two: {input_str}"

@mlflow.trace(span_type="func")
async def call_three(input_str: str, wait_period=10):
  await asyncio.sleep(wait_period) # Simulate a bottlenecked API call for 10s
  return f"call_three: {input_str}"

@mlflow.trace(name="Trace Test")
async def call_chain(input_str: str, wait_period:int):
  results = await asyncio.gather(
      call_one(input_str),
      call_two(input_str),
      call_three(input_str, wait_period)
  )
  return results

# Use 'await' directly on the function call when running this in a notebook.
final_result = await call_chain('first_call', 10)

print(final_result)

# Wrap functions in PyFunc API
Transform local function above into pyfunc. The main change is moving call_chain --> predict. Note that we have to use `predict` as the main entry point into the pyfunc.

NOTE: You will have to transform `predict` under the pyfunc to an `async def` if you want to test locally. There are four sections you'll note you have to call an `await` for local dev. Make sure to swap it back to `asyncio.run` for deployment. The reason you have to do this is because `asyncio.run` doesn't work in Jupyter notebooks but does for endpoints. 

In [0]:
import asyncio
import pandas as pd
import mlflow
import random

class LoggingAPICalls(mlflow.pyfunc.PythonModel):
  @mlflow.trace(span_type="func")
  async def call_one(self, input_str: str):
    # use this to mimic first API call
    raise Exception('This is broken')
    wait_time = random.randint(1, 15)
    await asyncio.sleep(wait_time)
    return f"call_one: {input_str} took {wait_time}s"
  
  @mlflow.trace(span_type="func")
  async def call_two(self, input_str: str):
    # use this to mimic second API call
    wait_time = random.randint(1, 15)
    await asyncio.sleep(wait_time)
    return f"call_two: {input_str} took {wait_time}s"
  
  @mlflow.trace(span_type="func")
  async def call_three(self, input_str: str):
    # use this to mimic third API call
    wait_time = random.randint(1, 15)
    await asyncio.sleep(wait_time)
    return f"call_three: {input_str} took {wait_time}s"

  @mlflow.trace(span_type="func")
  async def call_chain(self, input_str: str):
    """
      This async orchestrates all the APIs to be called. 
    """
    row_task = await asyncio.gather(
        self.call_one(input_str),
        self.call_two(input_str),
        self.call_three(input_str)
    )
    return row_task
  
  def predict(self, context, model_input):
  # if testing locally in notebook change to this:
  # async def predict(self, context, model_input):
    
    async def process_dataframe_concurrently(model_input: pd.DataFrame):
      """
      This async function creates coroutines for each row in the model_input. 
      """
      tasks = []
    
      # Iterate over the DataFrame rows to create a task for each one.
      # Crucially, we do NOT await inside this loop.
      for index, row in model_input.iterrows():
        input_str = row['input_str']
        
        # Create the coroutine for the row and add it to the tasks list.
        # This does not execute the function yet.
        task = self.call_chain(input_str)
        tasks.append(task)

      # The single 'await' here will wait for ALL tasks in the list to complete.
      all_tasks = await asyncio.gather(*tasks)
      
      return all_tasks

    # Run async task. Note that this is going to be blocked by the longest running call. 
    @mlflow.trace(name="Trace Test")
    def asyncio_run_coroutine(model_input):
    # if testing locally in notebook change to this:
    # async def asyncio_run_coroutine(model_input):
      """
      Trigger the coroutines and return the results. This will be the main IO we log. 
      """
      results = asyncio.run(process_dataframe_concurrently(model_input))
      # if testing locally in notebook change to this:
      # results = await process_dataframe_concurrently(model_input)

      return results
    
    try:
      final_results = asyncio_run_coroutine(model_input)
      # if testing locally in notebook change to this:
      # final_results = await asyncio_run_coroutine(model_input)

      ## CBXPY --> sub 2 seconds to solve 
    except Exception as e:
      final_results = f"Could not complete blah blah... {e}"

    return final_results 


In [0]:
model = LoggingAPICalls()

test_input = pd.DataFrame([{'input_str': 'payload_01'}])

# if testing locally in notebook run this:
test_output = await model.predict(None, test_input)

print(test_output)


In [0]:
# Wrap final model with I/O signatures for logging and registration
from mlflow.models import infer_signature
catalog = 'main'
schema = 'jon_cheung'
model_name='api_logging_pyfunc'

test_input = pd.DataFrame([{'input_str': 'payload_01'},
                          {'input_str': 'payload_02'}])

test_output = pd.DataFrame(['call_one: payload_01 took 3s',
                             'call_two: payload_01 took 1s',
                             'call_three: payload_01 took 4s'],
                            ['call_one: payload_02 took 8s',
                             'call_two: payload_02 took 3s',
                             'call_three: payload_02 took 5s'])

signature = infer_signature(test_input, test_output)

mlflow.pyfunc.log_model(name='api_pyfunc', 
                        python_model=LoggingAPICalls(), 
                        signature=signature,
                        registered_model_name=f'{catalog}.{schema}.{model_name}')

## Create a model serving endpoint.

We will create an endpoint using the mlflow SDK.

What we will enable is:
* AI Gateway Inference Tables - this will log the payload. Note that there is a 30-60min delay from inference request. 
* Real-time Tracing - the traces will show up in the mlflow experiment you define
* System tables - this will log usage. 


In [0]:
import mlflow
catalog = 'main'
schema = 'jon_cheung'
model_name='api_logging_pyfunc'
experiment_name = '/Users/jon.cheung@databricks.com/real-time-trace-logging'


if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)
current_experiment = mlflow.get_experiment_by_name(experiment_name)
experiment_id=current_experiment.experiment_id

experiment_id

In [0]:

from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
endpoint = client.create_endpoint(
    config={
        "name": "trace-api-calls",
        # This will enable logging inferences. The table name is the endpoint + _payload) 
        "ai_gateway": {
          "inference_table_config": {
            "enabled": True,
            "catalog_name": catalog,
            "schema_name": schema},
          "usage_tracking_config": {
            "enabled": True},
        },
        "config": {
          "served_entities": [
            {  
                "entity_name": f'{catalog}.{schema}.{model_name}',
                "entity_version": "7",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                # This will enable real-time tracing with results stashed into the mlflow experiment defined above
                "environment_vars": {"ENABLE_MLFLOW_TRACING": "true", 
                                     "MLFLOW_EXPERIMENT_ID": f"{experiment_id}", 
                                     "DATABRICKS_HOST": "https://e2-demo-field-eng.cloud.databricks.com/",
                                     "DATABRICKS_TOKEN": "{{secrets/development/jon_cheung_PAT}}"}
            },
          ],
          "traffic_config": {
            "routes": [
                {
                    "served_model_name": f"{model_name}-7",
                    "traffic_percentage": 100
                }
            ]
          } 
        }
    }
)

## Testing inference of served model endpoint from the above

In [0]:
import os
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get(scope = "development", key = "jon_cheung_PAT")


# TODO: Insert your serving endpoint here. 
model_serving_url = 'https://e2-demo-field-eng.cloud.databricks.com/serving-endpoints/trace-api-calls/invocations'

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = model_serving_url
    headers = {'Authorization': f'Bearer {os.environ.get("DATABRICKS_TOKEN")}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()
  
test_input = pd.DataFrame([{'input_str': 'payload_that will not break'} ])


outputs = score_model(test_input)

In [0]:
outputs